<!-- bootcamp-header: generated by tools/build_headers.py, edit the README timetable instead -->
# pandas

**Session:** Wednesday 7 October 2026, 10:30-12:30, room S.R.118  
**Tutors:** Loren Verreyen / Caroline Vandyck  
**Exercises:** [`13_EX_Pandas.ipynb`](https://github.com/mikekestemont/dtaantwerp26-27.github.io/blob/DTA_Bootcamp_2026_students/exercises/questions/13_EX_Pandas.ipynb) (solutions: [`13_SOL_Pandas.ipynb`](https://github.com/mikekestemont/dtaantwerp26-27.github.io/blob/DTA_Bootcamp_2026_students/exercises/solutions/13_SOL_Pandas.ipynb))  

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mikekestemont/dtaantwerp26-27.github.io/blob/DTA_Bootcamp_2026_students/notebooks/13_W3_Wed_Pandas.ipynb)

In [ ]:
# Run this cell only if you are working on Google Colab: it downloads the course
# material (notebooks and data) so that the file paths in this notebook work.
# On your own computer you can skip it.
!git clone --quiet --depth 1 --branch DTA_Bootcamp_2026_students https://github.com/mikekestemont/dtaantwerp26-27.github.io.git bootcamp
%cd bootcamp/notebooks

## Tables with `pandas`

Much of the data you will meet is not running text but a **table**: a spreadsheet of metadata, a CSV file of annotations, the output of a corpus tool. Python's standard tool for tables is the library **`pandas`**, and this session is a first, practical tour of it, on one dataset. You will not learn all of `pandas` in two hours, nobody does; you will learn the dozen operations that cover most of what a text analyst does with a table, and how to look up the rest.

After this session you can:

- load a CSV file into a `DataFrame` and inspect it;
- select columns and rows, and filter rows with a condition;
- count and summarise a column, and compare groups with `groupby()`;
- add a column computed from other columns;
- draw a bar chart, a line, a box plot or a scatter plot of a result, and save a table back to CSV.

## A first look

`pandas` is imported under the alias `pd` by universal convention. Its central object is the **`DataFrame`**: a table with named columns and numbered rows. The usual way to get one is to read a CSV file (*comma-separated values*: one row per line, values separated by commas, column names on the first line).

### The dataset

In 2016 Hannah Anderson and Matt Daniels of *The Pudding* took 2,000 Hollywood screenplays apart and counted, line by line, who actually gets to speak in the movies. Every character in every script got a row: how many words they say, what share of the film's dialogue that is, and, where IMDb could tell them, the age and the gender of the actor playing the part. The result is one of the more uncomfortable datasets in film studies: in a great many films, the women are barely in the room.

`screenplays.csv` holds their data for films released between 1929 and 2015. Each row is **one character in one film**, with these columns:

- `title`: the title of the film;
- `release_year`: the year the film was released;
- `character`: the name of the character, as it appears in the screenplay;
- `gender`: the gender of the actor playing the character, as coded from IMDb (a binary, because that is what IMDb offered; the researchers themselves call this an imperfect approach);
- `words`: the number of words the character speaks;
- `proportion_of_dialogue`: that character's share of the film's dialogue;
- `age`: the age of the actor when the film was released;
- `gross`: the film's box-office gross, in millions of dollars;
- `script_id`: an identifier for the screenplay.

In [ ]:
import pandas as pd

df = pd.read_csv('../data/screenplays.csv')
df.head()

`df` is the conventional name for a DataFrame. `.head()` shows the first five rows (`.tail()` the last five; give a number to see more). The bold numbers on the left are the **index**: the row labels, here just 0, 1, 2, ... Every column has a name and a type, and a first inspection is always the same four questions: how big, which columns, which types, anything missing?

In [ ]:
print(df.shape)         # (rows, columns)
print(len(df))          # rows
print(df.columns)

In [ ]:
df.info()

`.info()` answers all four at once. The types: `int64` and `float64` are numbers, `object` is text (or mixed). The *non-null* counts show that `age` is missing for about 4,800 characters and `gross` for about 3,700; missing values are shown as `NaN` (*not a number*) in the table, and `pandas` skips them when it computes averages.

`.describe()` gives the standard statistics of every numerical column in one go. Read it before you trust a dataset: the maximum `age` here is 2009, which is not an age but a typing error somewhere in IMDb, and it would quietly distort every average you compute on that column.

In [ ]:
df.describe()

A CSV file is not always comma-separated, and not always UTF-8: `pd.read_csv(path, sep=';', encoding='latin-1')` reads a semicolon-separated file in another encoding. If a file loads as one column, or with strange characters, these two arguments are the first thing to check.

## Columns and rows

A single column, selected with its name between square brackets, is a **`Series`**: one-dimensional, with the same index as the table. Most of what you do in `pandas` you do to a Series.

In [ ]:
words = df['words']
print(type(words))
words.head()

In [ ]:
print(words.mean(), words.min(), words.max())
print(words.median())

Several columns at once: a *list* of names inside the brackets (hence the double brackets). The result is again a DataFrame.

In [ ]:
df[['character', 'title', 'words']].head()

Rows are selected by position with `.iloc[]` (like a list) or by index label with `.loc[]`; with the default index both give the same here. `.loc[]` also takes a column name as a second argument, which is the way to read one cell:

In [ ]:
print(df.iloc[0])                 # the first row, as a Series
print(df.loc[0, 'character'])     # one cell: row 0, column character
df.iloc[10:15]                    # rows 10 to 14

## Filtering rows with a condition

The operation you will use most. A comparison on a column gives a Series of `True`/`False`, one per row, a **boolean mask**; putting that mask between the brackets of the DataFrame keeps the rows where it is `True`:

In [ ]:
df['words'] > 10000

In [ ]:
talkative = df[df['words'] > 10000]
print(len(talkative))
talkative[['character', 'title', 'words']]

Read `df[df['words'] > 10000]` as "the rows of `df` where `words` is above 10,000". Text columns compare with `==`, and a Series of strings has string methods behind `.str`, such as `.str.contains()` (which, like `re.findall()`, reads its argument as a pattern):

In [ ]:
women = df[df['gender'] == 'woman']
print(len(women))

star_wars = df[df['title'].str.contains('Star Wars')]
star_wars[['title', 'character', 'words']].head()

To combine conditions you need `&` (and) and `|` (or), **not** `and` and `or`, and every condition goes between **round brackets**. This is the place where `&` and `|` finally belong; they work element by element on the two masks, which is what `and` and `or` cannot do.

In [ ]:
df[(df['gender'] == 'woman') & (df['release_year'] >= 2010)].head()

In [ ]:
df[(df['release_year'] < 1940) | (df['release_year'] == 2015)][['title', 'release_year', 'character']].head()

`.isin()` tests membership in a list, which is shorter than several `|`:

In [ ]:
df[df['release_year'].isin([1994, 1995])].shape

### Class exercises

1. How many characters are played by women? How many of those speak more than 5,000 words? Show the character, title and words of every character in *Titanic* (the 1997 film), most words first (`.sort_values()` comes a little later; a filter and the three columns is enough for now).

In [ ]:
# your code here

## Counting and summarising

`.value_counts()` counts how often each value occurs in a column, most frequent first: it is `Counter` for a Series. `.unique()` lists the distinct values and `.nunique()` counts them.

In [ ]:
print(df['gender'].value_counts())
print(df['title'].nunique(), df['script_id'].nunique())     # not the same: some titles were filmed twice

`.sort_values()` sorts a Series or, with a column name, a whole DataFrame. `.idxmax()` gives the index label of the largest value, which combined with `.loc[]` answers "who?" questions:

In [ ]:
df.sort_values('words', ascending=False)[['character', 'title', 'words']].head()

In [ ]:
biggest = df['gross'].idxmax()
print(df.loc[biggest, 'title'], df.loc[biggest, 'gross'])

A trick you will use constantly: the **mean of a mask is a proportion**, because `True` counts as 1 and `False` as 0. The share of characters played by women, and the share of characters with more than 1,000 words, in one line each:

In [ ]:
print((df['gender'] == 'woman').mean())
print((df['words'] > 1000).mean())

## Comparing groups: `groupby()`

The question behind most tables is "does this differ between groups?" `groupby()` splits the table by the values of one column, and a summary method after it is applied to each group separately. Words spoken, by gender:

In [ ]:
df.groupby('gender')['words'].sum()

Read it left to right: *group the rows by `gender`, take the `words` column, add it up per group*. Men speak almost three times as many words as women in these 2,000 films. Any summary works after the column: `.mean()`, `.sum()`, `.count()`, `.max()`, and `.size()` for the number of rows in each group.

In [ ]:
print(df.groupby('gender')['words'].mean())
print(df.groupby('gender').size())

Two grouping columns give a group for every combination, as a list of two names. Words per film and gender, for the first films alphabetically:

In [ ]:
df.groupby(['title', 'gender'])['words'].sum().head(8)

### Class exercise

2. What is the average age of the actors per gender? How many characters does the data have per release year, for the years 2010 to 2015 (group, then filter or slice the result)? Which film has the most characters (`.groupby('title').size()` and `.idxmax()`)?

In [ ]:
# your code here

## New columns

A new column is made by assigning to a name that does not exist yet, and the value can be computed from other columns, row by row, without a loop: arithmetic, comparisons and `.str` methods all work on whole columns at once. Numbers have to be turned into strings with `.astype(str)` before they can be glued to text.

In [ ]:
df['decade'] = df['release_year'] // 10 * 10
df['is_woman'] = df['gender'] == 'woman'
df['film'] = df['title'] + ' (' + df['release_year'].astype(str) + ')'    # tells the two King Kongs apart

df[['film', 'decade', 'character', 'is_woman']].head()

When the computation needs a function of your own, `.apply()` calls it on every value of a column and collects the results. Here we label every character by the size of the part:

In [ ]:
def part_size(proportion):
    if proportion >= 0.2:
        return 'lead'
    if proportion >= 0.05:
        return 'supporting'
    return 'minor'

df['part'] = df['proportion_of_dialogue'].apply(part_size)
df['part'].value_counts()

Two more things about missing values: `.isna()` marks them (and `.isna().sum()` counts them per column), and `.dropna()` drops the rows that have any. Do the second only when you mean it: `df.dropna()` would throw away every character without a known age *or* gross, so name the column that matters.

In [ ]:
print(df.isna().sum())
with_age = df.dropna(subset=['age'])
print(len(df), len(with_age))

## Plots

A Series has a `.plot()` method, which draws it with the library `matplotlib`. `kind='bar'` gives a bar chart, the right choice for a result of `value_counts()` or `groupby()`. `matplotlib` itself is imported for the title and labels, and `plt.show()` displays the figure. The *Pudding* question, per decade: what share of all dialogue is spoken by women? Two `groupby()` sums, divided (two Series with the same index divide element by element):

In [ ]:
import matplotlib.pyplot as plt

words_by_women = df[df['is_woman']].groupby('decade')['words'].sum()
words_total = df.groupby('decade')['words'].sum()

(words_by_women / words_total).plot(kind='bar')
plt.title('Share of dialogue spoken by women, per decade')
plt.xlabel('decade')
plt.ylabel('share of words')
plt.show()

In [ ]:
df[df['age'] < 100]['age'].plot(kind='hist', bins=20)      # the filter drops the impossible ages
plt.title('Age of the actors')
plt.xlabel('age')
plt.show()

`kind='barh'` gives horizontal bars, which suit long labels; `kind='hist'` a histogram of a numerical column. Three more kinds answer questions that bars cannot.

A **line** is the default kind, and the right one for anything ordered in time. The same *Pudding* question per year rather than per decade:

In [ ]:
words_by_women_per_year = df[df['is_woman']].groupby('release_year')['words'].sum()
words_per_year = df.groupby('release_year')['words'].sum()

(words_by_women_per_year / words_per_year).plot()
plt.title('Share of dialogue spoken by women, per year')
plt.xlabel('year')
plt.ylabel('share of words')
plt.show()

The line is noisier than the bars (few films per year before 1970), but it shows the trend and its bumps in a way ten bars cannot.

A **box plot** shows the distribution of a numerical column per group: the line in the box is the median, the box holds the middle half of the values, the whiskers the rest, and the dots are outliers. `df.boxplot(column=..., by=...)` draws one box per group. Ages by gender, on the table without the impossible values:

In [ ]:
clean = df[df['age'] < 100]

clean.boxplot(column='age', by='gender')
plt.title('Age of the actors, by gender')
plt.suptitle('')          # removes the automatic second title
plt.ylabel('age')
plt.show()

Actresses are cast younger, and their ages are packed more tightly; that is the whole *Pudding* argument in one figure.

A **scatter plot** puts two numerical columns against each other, one dot per row, and is the way to see whether they are related. `alpha` makes the dots translucent, so that dense regions show as darker: with 23,000 rows that matters.

In [ ]:
clean.plot(kind='scatter', x='age', y='words', alpha=0.1)
plt.title('Words spoken against age of the actor')
plt.show()

The big parts are spread over all ages; most characters, of any age, speak few words. That is as far as plotting goes in this course; the module on data visualisation takes it from here.

## Saving

`.to_csv()` writes a DataFrame back to a file. `index=False` leaves out the row numbers, which you rarely want in the file. Results belong in a folder of their own, not among the notebooks: `os.makedirs()` creates one (and does nothing if it exists already).

In [ ]:
import os

os.makedirs('output', exist_ok=True)

leads = df[df['part'] == 'lead'][['film', 'character', 'gender', 'words']]
leads.to_csv('output/leads.csv', index=False)

pd.read_csv('output/leads.csv').head(3)

## Making a table yourself

Tables do not only come from files. A list of dictionaries, one per row with the column names as keys (the shape you met in the session on dictionaries), turns into a DataFrame directly, and so does a `Counter` via its `.items()`:

In [ ]:
from collections import Counter

rows = [{'title': 'Emma', 'author': 'Austen', 'year': 1815},
        {'title': 'Dracula', 'author': 'Stoker', 'year': 1897},
        {'title': 'Middlemarch', 'author': 'Eliot', 'year': 1871}]
novels = pd.DataFrame(rows)
print(novels)

counts = Counter('the cat sat on the mat with the hat'.split())
frequencies = pd.DataFrame(counts.items(), columns=['word', 'count']).sort_values('count', ascending=False)
print(frequencies)

### Class exercise

3. Read `alice.txt`, tokenise it with `re.findall(r'[a-z]+', text.lower())`, count the tokens with a `Counter`, turn the counter into a DataFrame with the columns `word` and `count`, add a column `length` with the length of each word, and show the ten most frequent words of more than six letters.

In [ ]:
import re

# your code here

## Common mistakes

Four things that go wrong in everyone's first week with `pandas`.

In [ ]:
# This cell produces an error on purpose: 'and' does not work on columns; use & with round brackets
df[df['gender'] == 'woman' and df['release_year'] > 2010]

In [ ]:
# This cell produces an error on purpose: & without brackets around each condition
df[df['gender'] == 'woman' & df['release_year'] > 2010]

In [ ]:
# This cell produces an error on purpose: a column name that does not exist (case matters)
df['Age'].mean()

In [ ]:
# Not an error, but a silent trap: a method returns a new table; the original is unchanged unless you assign
df.sort_values('words')
print(df['words'].head(3))          # not sorted
df_sorted = df.sort_values('words')
print(df_sorted['words'].head(3))   # sorted

## Quick reference

| Task | Code |
| --- | --- |
| load, inspect | `pd.read_csv(path)`, `.head()`, `.shape`, `.columns`, `.info()`, `.describe()` |
| one column, several | `df['col']`, `df[['a', 'b']]` |
| rows | `df.iloc[i]`, `df.loc[i, 'col']`, `df[mask]` |
| conditions | `df['col'] > 5`, `==`, `.str.contains()`, `.isin([...])`, `(a) & (b)`, `(a) \| (b)` |
| summarise | `.mean()`, `.sum()`, `.min()`, `.max()`, `.median()`, `.value_counts()`, `.unique()`, `.nunique()` |
| sort, locate | `.sort_values('col', ascending=False)`, `.idxmax()` |
| groups | `df.groupby('col')['other'].mean()`, `.groupby(['a', 'b'])` |
| new column | `df['new'] = df['a'] + df['b']`, `df['col'].apply(function)`, `.str.` methods |
| missing values | `.isna().sum()`, `.dropna(subset=['col'])` |
| plot, save | `.plot(kind='bar')`, `.plot()` (line), `.plot(kind='scatter', x=, y=)`, `.boxplot(column=, by=)`, `plt.title()`, `plt.show()`, `.to_csv(path, index=False)` |

## References

- [10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html): the official quick tour
- [pandas cheat sheet](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf) (PDF)
- [Film Dialogue from 2,000 screenplays, Broken Down by Gender and Age](https://pudding.cool/2017/03/film-dialogue/): the *Pudding* article the dataset comes from